# 07 — Which pseudo-landmarks can we trust? Landmark-selection rules for label-free recovery on an unseen subject

No training. Uses the LOSO models already saved for every fold:
`v6b/loso_route_c_<sub>.pt`, `v6b/loso_L28_<sub>.pt` (InternViT-6B) and `v6/loso_route_<sub>.pt`, `v6/loso_L20_<sub>.pt` (DINOv2-giant).

Recovery = SCORE-style one-step coordinate recovery (moment matching → CSLS → 64 landmarks → margin-weighted orthogonal Procrustes with identity regularisation, rho 0.1) **in the route_c space**. Only the landmark rule changes; the final score is always route_c after recovery, so differences come from landmark choice alone.

| Rule | Role |
|---|---|
| `score` | SCORE: MNN pairs of route_c, top 64 by margin (baseline; should give 44.7 mean) |
| `random` | 64 random MNN pairs (mean of 20 draws) — control |
| `ens` | MNN + margin on the averaged z-scored CSLS matrix of all 4 models — **key control** (any gain from consensus must beat plain ensembling) |
| `cons_intern` | route_c MNN pairs ranked by agreement of L28 (top-3), then margin |
| `cons_dino` | … agreement of the two DINOv2 models (independent backbone) |
| `cons_all` | … agreement of L28 + both DINOv2 models (0–3 votes) |
| `hungarian` | MNN pairs that also belong to the optimal one-to-one assignment, then margin — control (uses closed-gallery assumption) |
| `geometry` | margin × neighbourhood consistency with 16 provisional anchors |
| `oracle` | 64 correct pairs (uses the answer key) — **upper bound only, not a method** |

Diagnostics per fold: Precision@64, recovered Top-1/Top-5, effective rank of the landmark set, precision by number of votes, error correlation between models.
The answer key is used only after selection, for evaluation (and for `oracle`).

**GPU:** A100 or L4 if LOSO model files are missing (they are retrained here with the 06/06b recipe, ~1 min per fold); otherwise any GPU, ~5 min.

In [ ]:
# 0. Setup
import os, sys, gc, json, math, time, random, shutil, glob, copy, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")
DCACHE = f"{DRIVE}/cache"
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")
OUT    = f"{DRIVE}/v7"
V6, V6B = f"{DRIVE}/v6", f"{DRIVE}/v6b"
for d in [DRIVE, DCACHE, ROOT, LOCAL, OUT]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
SMOKE = os.environ.get("EEG_SMOKE") == "1"           # tiny synthetic run for code checking only

import numpy as np
for pkg, pip_name in [("huggingface_hub", "huggingface_hub"), ("pandas", "pandas"), ("scipy", "scipy")]:
    if not SMOKE and importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8      # bfloat16 needs Ampere or newer (not T4)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| bf16:", AMP, "| torch", torch.__version__)
print(f"Drive free: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB (features need ~1.2 GB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

In [ ]:
# 1. Image list (same order as notebooks 04/05) and the validation split
META = f"{DCACHE}/meta_v6.pt"
if not os.path.exists(META):
    import tarfile
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    torch.save({"train_files": train_files, "test_files": test_files,
                "train_rel": [os.path.relpath(idx_train[f], ROOT) for f in train_files],
                "test_rel": [os.path.relpath(idx_test[f], ROOT) for f in test_files],
                "n_concepts": len(set(meta["train_img_concepts"]))}, META)
M = torch.load(META, weights_only=False)
N_TR, N_TE = len(M["train_files"]), len(M["test_files"])
PER = N_TR // M["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)              # last image of every training concept (same as 01–05)
print(f"train images {N_TR} | test images {N_TE} | concepts {M['n_concepts']} | val images {IS_VAL.sum()}")

In [ ]:
# 2. Cached image features of both backbones (written by notebooks 06 / 06b) and z-scored targets (training statistics)
def load_feats(b):
    p = f"{DCACHE}/feat6_{b}.pt"
    if os.path.exists(p): return torch.load(p, weights_only=False)
    parts = f"{DCACHE}/feat6_{b}_parts"                     # 06b saved chunks; the merged file may not have synced
    assert os.path.isdir(parts), f"neither {p} nor {parts} found"
    def merge(tag):
        files = sorted(glob.glob(f"{parts}/{tag}_*.pt")); print(f"  {b} {tag}: {len(files)} chunks")
        ch = [torch.load(f) for f in files]
        return {k: torch.cat([c[k] for c in ch]) for k in ch[0]}
    tr, te = merge("train"), merge("test")
    layers = sorted({l for l, _ in tr})
    assert len(tr[(layers[0], "mean")]) == N_TR and len(te[(layers[0], "mean")]) == N_TE, "chunks incomplete"
    return {"layers": layers, "tr": tr, "te": te}

FE = {b: load_feats(b) for b in ["internvit6b", "dinov2g"]}
for b, d in FE.items(): print(b, "layers", d["layers"], "dim", d["tr"][(d["layers"][0], "mean")].shape[1])

def target_te(b, layer):
    tr, te = FE[b]["tr"][(layer, "mean")].float(), FE[b]["te"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((te - m) / s).to(device)

CH63 = ["Fp1","Fp2","AF7","AF3","AFz","AF4","AF8","F7","F5","F3","F1","F2","F4","F6","F8","FT9","FT7","FC5","FC3","FC1",
        "FCz","FC2","FC4","FC6","FT8","FT10","T7","C5","C3","C1","Cz","C2","C4","C6","T8","TP9","TP7","CP5","CP3","CP1",
        "CPz","CP2","CP4","CP6","TP8","TP10","P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8",
        "O1","Oz","O2"]
_, N_CH_ALL, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
def load_test(sub):
    x = np.load(local_copy(f"{sub}_test_z16.npy"), mmap_mode="r")
    return torch.from_numpy(np.array(x)).to(device)
def load_train(sub):
    return torch.from_numpy(np.array(np.load(local_copy(f"{sub}_train_z16.npy"), mmap_mode="r"))).to(device)
def target_tr(b, layer):
    tr = FE[b]["tr"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((tr - m) / s).to(device)
VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device)


In [ ]:
# 4. Model: EEGProject (MLP with residual bottleneck, as in Shallow Alignment) + linear image projector(s)
#    Single-layer target, or a learned softmax routing over several layers (global SAMGA-style prior, no subject residual).
EMB = 1024

class EEGProject(nn.Module):
    def __init__(self, n_in, d=EMB, drop=0.3):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d, d), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, x):
        h = self.inp(x.flatten(1)); return self.norm(h + self.res(h))

class ImageSide(nn.Module):
    def __init__(self, dims, init_logits=None):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, EMB) for d in dims])
        self.logits = nn.Parameter(torch.tensor(init_logits, dtype=torch.float) if init_logits is not None else torch.zeros(len(dims)),
                                   requires_grad=len(dims) > 1)
    def weights(self): return F.softmax(self.logits, 0)
    def forward(self, feats):                   # feats: list of (N, d_l), same rows
        w = self.weights()
        return sum(w[i] * p(f) for i, (p, f) in enumerate(zip(self.proj, feats)))

class Aligner(nn.Module):
    def __init__(self, n_in, dims, init_logits=None):
        super().__init__()
        self.eeg, self.img = EEGProject(n_in), ImageSide(dims, init_logits)
        self.log_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    def scale(self): return self.log_scale.exp().clamp(max=100)

def clip_loss(ze, zi, scale):
    ze, zi = F.normalize(ze, dim=-1), F.normalize(zi, dim=-1)
    logits = scale * ze @ zi.T; y = torch.arange(len(ze), device=ze.device)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

@torch.no_grad()
def embed_eeg(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
            out.append(F.normalize(model.eeg(X[i:i + bs].float()).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_img(model, feats):
    model.eval(); return F.normalize(model.img(feats).float(), dim=-1)

@torch.no_grad()
def metrics(sim):
    # sim: queries x candidates, the correct candidate of query i is candidate i
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

EPOCHS, BATCH, LR, WD, EVAL_EVERY = (60, 1024, 1e-4, 1e-4, 5) if not SMOKE else (3, 64, 1e-3, 1e-4, 1)
ROUTE_INIT = [-2.0, -1.0, 0.0, -1.0, -2.0]

def train(X, img_of, T_tr, val_sets, seed=0, log=None, init_logits=None):
    # X: EEG trials, img_of[i]: training image of trial i, T_tr: list of target matrices (N_TR x d_l)
    # val_sets: list of (EEG rows, image rows) used for model selection (held-out training images)
    set_seed(seed)
    model = Aligner(X[0].numel(), [t.shape[1] for t in T_tr], init_logits).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    tr_rows = torch.where(~torch.from_numpy(IS_VAL).to(device)[img_of])[0]
    steps = EPOCHS * math.ceil(len(tr_rows) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_rows[torch.randperm(len(tr_rows), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            imgs = img_of[b]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
                ze = model.eeg(X[b].float()); zi = model.img([t[imgs] for t in T_tr])
            loss = clip_loss(ze.float(), zi.float(), model.scale())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            zi_val = embed_img(model, [t[VAL] for t in T_tr])
            va = float(np.mean([metrics(embed_eeg(model, X[er]) @ zi_val[ir].T)["top1"] for er, ir in val_sets]))
            if log: log(ep, va, model)
            if va > best: best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_ep, best
@torch.no_grad()
def metrics(sim):
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

AMP = AMP if "AMP" in globals() else False
# the four LOSO models of every fold: (name, folder, file tag, backbone, layers)
MODELS = [("intern_route", V6B, "route_c", "internvit6b", [20, 24, 28, 32, 36]),
          ("intern_L28",   V6B, "L28",     "internvit6b", [28]),
          ("dino_route",   V6,  "route",   "dinov2g",     [20, 24, 28, 32, 36]),
          ("dino_L20",     V6,  "L20",     "dinov2g",     [20])]
PRIMARY = "intern_route"

MODEL_LOCAL = f"{LOCAL}/models"; os.makedirs(MODEL_LOCAL, exist_ok=True)
ROUTE_INIT5 = [-2.0, -1.0, 0.0, -1.0, -2.0]

def model_path(folder, tag, sub):
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    return loc if os.path.exists(loc) else f"{folder}/loso_{tag}_{sub}.pt"

def retrain(folder, tag, sub, backbone, layers):
    """same recipe as notebooks 06/06b cell 6 (9 source subjects, 63 ch, seed 0, best epoch on source validation images)"""
    t0 = time.time(); srcs = [s for s in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:3]) if s != sub]
    X = torch.empty((len(srcs) * N_TR, N_CH_ALL, N_T), dtype=torch.float16, device=device)
    for k, s_ in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = load_train(s_)
    img_of = torch.arange(len(X), device=device) % N_TR
    vsets = [(k * N_TR + VAL, torch.arange(len(VAL), device=device)) for k in range(min(3, len(srcs)))]
    T_tr = [target_tr(backbone, l) for l in layers]
    m, ep, va = train(X, img_of, T_tr, vsets, 0, None, ROUTE_INIT5[-len(layers):] if len(layers) > 1 else None)
    del X, T_tr; free_gpu()
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    torch.save(m.state_dict(), loc)
    try: shutil.copy(loc, f"{folder}/loso_{tag}_{sub}.pt")
    except Exception as e: print("  (copy to Drive failed:", e, ")")
    print(f"  retrained {tag} {sub}: best epoch {ep}, val {va:.1f} ({(time.time()-t0)/60:.1f} min)", flush=True)

def load_model(folder, tag, sub, backbone, layers):
    if not os.path.exists(model_path(folder, tag, sub)): retrain(folder, tag, sub, backbone, layers)
    dims = [FE[backbone]["tr"][(l, "mean")].shape[1] for l in layers]
    m = Aligner(N_CH_ALL * N_T, dims).to(device)
    m.load_state_dict(torch.load(model_path(folder, tag, sub), map_location=device)); m.eval()
    return m

missing = [(n, sub) for n, folder, tag, bb, layers in MODELS for sub in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2])
           if not os.path.exists(model_path(folder, tag, sub))]
print(f"{len(missing)} LOSO model files missing -> retrained in cell 4 (~1 min each on A100/L4, ~4 min on T4):", missing)


In [ ]:
# 3. Recovery with a pluggable landmark rule (SCORE formulation, arXiv 2608.19134; same code as notebooks 06/06b)
CSLS_K, N_LAND, RHO, N_ANCH = 10, 64, 0.1, 16
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon, spearmanr

@torch.no_grad()
def csls(S, k=CSLS_K):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

def moment_match(Z, T):
    return F.normalize((Z - Z.mean(0)) / (Z.std(0) + 1e-6) * T.std(0) + T.mean(0), dim=-1)

def mnn_pairs(S):
    """mutual nearest neighbours of a query x candidate matrix; returns queries, candidates, margin of the query row"""
    top2 = S.topk(2, dim=1); j = top2.indices[:, 0]; i = torch.arange(len(S), device=S.device)
    mnn = S.argmax(0)[j] == i
    margin = (top2.values[:, 0] - top2.values[:, 1]).clamp(min=1e-4)
    q = torch.where(mnn)[0]
    return q, j[q], margin[q]

def pair_margin(S, q, c):
    """margin of candidate c over the best other candidate in row q (works for pairs that are not the row's top-1)"""
    s = S[q, c]; S2 = S[q].clone(); S2[torch.arange(len(q)), c] = -1e9
    return (s - S2.max(1).values).clamp(min=1e-4)

@torch.no_grad()
def procrustes(Zm, T, q, c, w):
    w = len(q) * w / w.sum()
    Xl, Yl = Zm[q], T[c]
    wn = w / w.sum(); muX = (wn[:, None] * Xl).sum(0); muY = (wn[:, None] * Yl).sum(0)
    C = ((Xl - muX) * w[:, None]).T @ (Yl - muY)
    Mx = C + RHO * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device)
    U, _, Vh = torch.linalg.svd(Mx.double()); R = (U @ Vh).float()
    return csls(F.normalize((Zm - muX) @ R + muY, dim=-1) @ T.T)

def eff_rank(X):
    s = torch.linalg.svdvals((X - X.mean(0)).double()); p = s / s.sum()
    return float(torch.exp(-(p * torch.log(p + 1e-12)).sum()))

def top_k_hit(S, q, c, k=3):
    return (S[q].topk(k, dim=1).indices == c[:, None]).any(1)

def take(q, c, key_main, key_tie, n=N_LAND):
    """rank pairs by key_main (desc), ties by key_tie (desc); keep n"""
    order = sorted(range(len(q)), key=lambda t: (-float(key_main[t]), -float(key_tie[t])))[:n]
    o = torch.tensor(order, device=q.device, dtype=torch.long)
    return q[o], c[o]


In [ ]:
# 4. Run all rules on the 10 LOSO folds
FOLDS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
N_RANDOM = 20 if not SMOKE else 3
rows, vote_rows, err_rows = [], [], []

for sub in FOLDS:
    Xt = load_test(sub)
    emb = {}
    for name, folder, tag, bb, layers in MODELS:
        m = load_model(folder, tag, sub, bb, layers)
        Z = embed_eeg(m, Xt); T = embed_img(m, [target_te(bb, l) for l in layers])
        Zm = moment_match(Z, T)
        emb[name] = dict(Z=Z, T=T, Zm=Zm, S0=Z @ T.T, S=csls(Zm @ T.T))
        del m; free_gpu()
    P = emb[PRIMARY]; Zm, T, S = P["Zm"], P["T"], P["S"]
    truth = torch.arange(len(S), device=device)
    others = [n for n in emb if n != PRIMARY]

    # error correlation between models (top-1 on the moment-matched CSLS matrices)
    pred = {n: e["S"].argmax(1) for n, e in emb.items()}
    for a in emb:
        for b in emb:
            if a < b:
                wa, wb = pred[a] != truth, pred[b] != truth
                both = (wa & wb).float().mean().item()
                same = ((pred[a] == pred[b]) & wa).float().mean().item()
                err_rows.append(dict(subject=sub, a=a, b=b, both_wrong=both * 100, same_wrong=same * 100,
                                     same_wrong_given_both=same / max(both, 1e-9) * 100))

    q0, c0, mg0 = mnn_pairs(S)                                   # route_c MNN pool
    votes = {n: top_k_hit(emb[n]["S"], q0, c0, 3).float() for n in others}
    v_intern = votes["intern_L28"]; v_dino = votes["dino_route"] + votes["dino_L20"]; v_all = v_intern + v_dino
    for t in range(len(q0)):
        vote_rows.append(dict(subject=sub, votes=int(v_all[t]), correct=bool(q0[t] == c0[t])))

    def ens_matrix():
        zs = [(e["S"] - e["S"].mean(1, keepdim=True)) / (e["S"].std(1, keepdim=True) + 1e-6) for e in emb.values()]
        return sum(zs) / len(zs)

    rules = {}
    rules["score"]       = take(q0, c0, mg0, mg0)
    rules["cons_intern"] = take(q0, c0, v_intern, mg0)
    rules["cons_dino"]   = take(q0, c0, v_dino, mg0)
    rules["cons_all"]    = take(q0, c0, v_all, mg0)
    qe, ce, mge = mnn_pairs(ens_matrix()); rules["ens"] = take(qe, ce, mge, mge)
    r_, c_ = linear_sum_assignment(-S.double().cpu().numpy()); assign = torch.full((len(S),), -1, device=device)
    assign[torch.as_tensor(r_, device=device)] = torch.as_tensor(c_, device=device)
    rules["hungarian"] = take(q0, c0, (assign[q0] == c0).float(), mg0)
    # neighbourhood geometry: 16 provisional anchors = strongest MNN pairs; Spearman of similarity profiles (anchor itself excluded)
    qa, ca = take(q0, c0, mg0, mg0, N_ANCH)
    sim_q = Zm @ Zm[qa].T; sim_c = T @ T[ca].T
    geo = []
    for t in range(len(q0)):
        keep = (qa != q0[t])
        a_ = sim_q[q0[t], keep].cpu().numpy(); b_ = sim_c[c0[t], keep].cpu().numpy()
        geo.append(spearmanr(a_, b_).statistic if keep.sum() > 2 else 0.0)
    geo = torch.tensor(np.nan_to_num(geo), device=device, dtype=torch.float)
    rules["geometry"] = take(q0, c0, mg0 * geo.clamp(min=0), mg0)
    oi = torch.argsort(-S.diag())[:N_LAND]; rules["oracle"] = (oi, oi)    # 64 correct pairs, most confident first (uses the answer key)

    def evaluate(rule, q, c):
        w = pair_margin(S, q, c)
        Srec = procrustes(Zm, T, q, c, w); mt = metrics(Srec)
        return dict(subject=sub, rule=rule, n_land=len(q), precision=float((q == c).float().mean() * 100),
                    eff_rank=eff_rank(Zm[q]), rec_top1=mt["top1"], rec_top5=mt["top5"],
                    plain_top1=metrics(P["S0"])["top1"], csls_top1=metrics(csls(P["S0"]))["top1"])
    for rule, (q, c) in rules.items(): rows.append(evaluate(rule, q, c))
    g = torch.Generator(device="cpu").manual_seed(0)
    rnd = [evaluate("random", q0[i], c0[i]) for i in [torch.randperm(len(q0), generator=g)[:N_LAND].to(device) for _ in range(N_RANDOM)]]
    rows.append({k: (np.mean([r[k] for r in rnd]) if isinstance(rnd[0][k], float) else rnd[0][k]) for k in rnd[0]})

    sel = {r["rule"]: r for r in rows if r["subject"] == sub}
    print(f"{sub}: MNN pool {len(q0)} | " + " | ".join(f"{k} {v['rec_top1']:.1f} (p {v['precision']:.0f})"
          for k, v in sel.items()), flush=True)
    del emb, Xt; free_gpu()

R = pd.DataFrame(rows); R.to_csv(f"{OUT}/landmark_rules.csv", index=False)
V = pd.DataFrame(vote_rows); V.to_csv(f"{OUT}/landmark_votes.csv", index=False)
E = pd.DataFrame(err_rows); E.to_csv(f"{OUT}/error_correlation.csv", index=False)


In [ ]:
# 5. Summary
order = ["oracle", "score", "random", "ens", "cons_intern", "cons_dino", "cons_all", "hungarian", "geometry"]
base = R[R.rule == "score"].set_index("subject")
out = []
for rule in order:
    g = R[R.rule == rule].set_index("subject").loc[base.index]
    d = g.rec_top1 - base.rec_top1
    p = wilcoxon(d).pvalue if rule != "score" and (d != 0).any() else float("nan")
    out.append(dict(rule=rule, precision=g.precision.mean(), rec_top1=g.rec_top1.mean(), rec_sd=g.rec_top1.std(),
                    worst=g.rec_top1.min(), vs_score=d.mean(), wins=int((d > 0).sum()), p_wilcoxon=p,
                    eff_rank=g.eff_rank.mean(), rec_top5=g.rec_top5.mean()))
S_ = pd.DataFrame(out).round(2)
print("Recovered Top-1 on the unseen subject (10 LOSO folds), route_c space, 64 landmarks")
print(f"reference: plain {base.plain_top1.mean():.1f} | CSLS {base.csls_top1.mean():.1f}")
print(S_.to_string(index=False))
S_.to_csv(f"{OUT}/landmark_rules_summary.csv", index=False)

print("\nPrecision of route_c MNN pairs by number of agreeing models (L28, DINOv2 route, DINOv2 L20; top-3)")
print(V.groupby("votes").correct.agg(["mean", "count"]).assign(mean=lambda x: (x["mean"] * 100).round(1)).to_string())

print("\nError overlap between models (% of test queries; mean over folds)")
print(E.groupby(["a", "b"])[["both_wrong", "same_wrong", "same_wrong_given_both"]].mean().round(1).to_string())

print("\nSpearman over subjects: landmark precision vs recovery gain (score rule)")
print(spearmanr(base.precision, base.rec_top1 - base.plain_top1))
print("\nPer subject, recovered Top-1:")
print(R.pivot(index="subject", columns="rule", values="rec_top1")[order].round(1).to_string())

try:
    from google.colab import drive; drive.flush_and_unmount(); print("Drive flushed: all files uploaded")
except Exception as e: print("flush skipped:", e)
